**Step 1: Identify the Task**

*Task*: Guide a self-driving trailer across Kenya’s highway network (Nairobi, Central Highlands, Rift Valley, and Lake Victoria Basin regions) from Nairobi to Kisumu using both distance-based search knowledge and domain routing rules.

*Goal*: Determine the lowest-distance (km) valid route while accounting for highway connectivity, search-tree paths, and operational/road-condition rules.

**Step 2: Assemble Relevant Knowledge**

*Prior Exercise Artifacts*:

- Directed highway graph with exact Google Maps / search distances (km) across 28 towns/nodes (e.g., Nairobi -> Ruaka at 15.5 km, Ruaka -> Limuru at 21.0 km, Nakuru -> Kericho at 107.0 km, Kericho -> Ahero at 60.9 km, Ahero -> Kisumu at 23.5 km, yielding the baseline optimal path of 357.0 km).

- Unfolded search tree expanding all distinct paths from Nairobi to Kisumu without converging cycles.

*Retrieved Domain & Corridor Knowledge (APIs / Web Scraping / Official Sources)*:

- Kenya National Highways Authority (KeNHA) corridor classifications and regional road management data (such as the Northern Corridor A8/A104 and B1/A1 links across Rift Valley and Nyanza/Lake Victoria Basin).
- Practical routing context: standard A104/B1 corridors via Nakuru and Kericho are paved major highways, whereas alternative routes (via Eldoret/Kakamega or via Narok/Bomet/Kisii/Awach) represent longer or distinct regional detours.
- Data Acquisition Note (APIs & Scraping): In a full implementation script/notebook (e.g., fetch_kb_data.py), you can pull live or batch distances via a routing/distance API (such as Google Distance Matrix / Routes API or OpenRouteService) and scrape or parse KeNHA corridor advisories/notices into a JSON/CSV knowledge store before loading into the KB.

**Step 3: Define the Vocabulary**

*Constants (Entities & Values)*: Towns (Nairobi, Ruaka, Limuru, Naivasha, Gilgil, Nakuru, Kericho, Ahero, Kisumu, Eldoret, Narok, etc.), status values (Active, Blocked, Caution, Paved), and numeric distances (km).

*Predicates / Relations (Represented in Python):*

- Road(u, v, dist_km): Directed highway edge from town u to town v with distance dist_km.

- OptimalRoute(start, goal, path, cost_km): Derived shortest distance path and total cost from start to goal.

- RoadStatus(u, v, status): Current operational condition of edge (u, v).

- TrailerRule(condition, action): Domain rule governing trailer routing decisions.

In [2]:
import os
import re
import heapq
from typing import Dict, List, Tuple, Optional
import requests
from bs4 import BeautifulSoup

# Step 4: Encode General Knowledge (Graph, Baseline UCS Reference, Domain Rules)
highway_graph: Dict[str, Dict[str, float]] = {
    "Nairobi": {"Gitaru": 22.3, "Ruaka": 15.5, "Kiambu": 15.5, "Ruiru": 24.6, "Maai-Mahiu": 53.8},
    "Gitaru": {"Limuru": 20.1},
    "Ruaka": {"Limuru": 21.0},
    "Kiambu": {"Limuru": 25.7},
    "Ruiru": {"Thika": 21.4},
    "Maai-Mahiu": {"Narok": 89.1},
    "Limuru": {"Naivasha": 56.1},
    "Naivasha": {"Gilgil": 30.4},
    "Gilgil": {"Nakuru": 42.6},
    "Nakuru": {"Timboroa": 94.5, "Eldama-Ravine": 63.1, "Kericho": 107.0},
    "Timboroa": {"Eldoret": 61.7},
    "Eldama-Ravine": {"Eldoret": 108.0},
    "Eldoret": {"Kapsabet": 46.0, "Kakamega": 106.0},
    "Kapsabet": {"Kisumu": 80.3},
    "Kakamega": {"Kisumu": 53.4},
    "Thika": {"Sagana": 63.9, "Othaya": 86.8},
    "Sagana": {"Nyahururu": 156.0},
    "Othaya": {"Nyeri": 16.9},
    "Nyeri": {"Nyahururu": 104.0},
    "Nyahururu": {"Nakuru": 63.3},
    "Kericho": {"Ahero": 60.9},
    "Narok": {"Bomet": 81.0},
    "Bomet": {"Kisii": 82.4},
    "Kisii": {"Homa-Bay": 52.0, "Awach": 77.4},
    "Homa-Bay": {"Ahero": 84.5},
    "Awach": {"Ahero": 14.1},
    "Ahero": {"Kisumu": 23.5},
    "Kisumu": {},
}

CORRIDOR_TOWNS = [
    "Nairobi", "Gitaru", "Ruaka", "Kiambu", "Ruiru", "Maai-Mahiu", "Maai Mahiu",
    "Limuru", "Naivasha", "Gilgil", "Nakuru", "Timboroa", "Eldama-Ravine", "Eldama Ravine",
    "Eldoret", "Kapsabet", "Kakamega", "Thika", "Sagana", "Othaya", "Nyeri",
    "Nyahururu", "Kericho", "Narok", "Bomet", "Kisii", "Homa-Bay", "Homa Bay",
    "Awach", "Ahero", "Kisumu",
]

corridor_metadata = {
    "authority": "Kenya National Highways Authority (KeNHA)",
    "corridors_covered": ["Northern Corridor / A8 / A104", "Rift Valley & Lake Victoria Basin links"],
    "baseline_optimal_path": [
        "Nairobi", "Ruaka", "Limuru", "Naivasha", "Gilgil",
        "Nakuru", "Kericho", "Ahero", "Kisumu",
    ],
    "baseline_optimal_cost_km": 357.0,
}

road_conditions: Dict[Tuple[str, str], Dict[str, object]] = {}

trailer_rules = {
    "Blocked": "Exclude segment from candidate search paths; reroute via next lowest-cost path.",
    "Caution": "Keep segment in search but flag extra trailer braking/stability caution.",
    "Default": "Traverse segment using shortest cumulative distance (Uniform Cost Search).",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; TrailerRoutingKBAgent/1.0; +https://example.org/bot)"
}


def normalize_town_name(name: str) -> str:
    return (
        name.strip()
        .replace("Maai Mahiu", "Maai-Mahiu")
        .replace("Eldama Ravine", "Eldama-Ravine")
        .replace("Homa Bay", "Homa-Bay")
    )

In [ ]:
#Step 5: Add New Facts to the Knowledge Base
def show_kb() -> None:
    print("\nCurrent Trailer Routing Knowledge Base")
    print(f"Total Origin Towns: {len(highway_graph)} | Corridor Authority: {corridor_metadata['authority']}")
    print("Active Road Condition Facts:")
    if not road_conditions:
        print("  (None — all graph edges operating at nominal distance cost)")
    for (u, v), info in road_conditions.items():
        print(
            f"  {u} -> {v} : {info['status']} (+{info.get('penalty_km', 0)} km) "
            f"| {info['reason']}"
        )
    print("Domain Rules:")
    for rule_key, action in trailer_rules.items():
        print(f"  {rule_key:8} -> {action}")

def add_road_fact(
    src: str,
    dst: str,
    status: str,
    reason: str = "Reported update",
    penalty_km: float = 0.0,
    source: str = "Manual/KB",
) -> None:
    """Programmatically add or update a road condition fact in the KB."""
    u = normalize_town_name(src)
    v = normalize_town_name(dst)
    norm_status = status.strip().capitalize()
    road_conditions[(u, v)] = {
        "status": norm_status,
        "reason": reason.strip(),
        "penalty_km": float(penalty_km),
        "source": source,
    }
    print(f"Added fact: Road({u} -> {v}) = {norm_status} ({reason}) [{source}]")

def update_edge_distance(src: str, dst: str, new_distance_km: float, source: str = "Google Maps Routes API") -> None:
    u = normalize_town_name(src)
    v = normalize_town_name(dst)
    if u in highway_graph and v in highway_graph[u]:
        old_dist = highway_graph[u][v]
        highway_graph[u][v] = round(float(new_distance_km), 2)
        print(f"[KB Distance Updated] {u} -> {v}: {old_dist}km -> {highway_graph[u][v]}km ({source})")

def fetch_google_maps_traffic_distance_km(
    origin: str,
    destination: str,
    api_key: Optional[str] = None,
    timeout: int = 10,
) -> Optional[Tuple[float, int]]:
    key = api_key or os.getenv("GOOGLE_MAPS_API_KEY")
    if not key:
        return None
    url = "https://routes.googleapis.com/directions/v2:computeRoutes"
    payload = {
        "origin": {"address": f"{normalize_town_name(origin)}, Kenya"},
        "destination": {"address": f"{normalize_town_name(destination)}, Kenya"},
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_AWARE",
        "units": "METRIC",
    }
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": key,
        "X-Goog-FieldMask": "routes.distanceMeters,routes.duration",
    }

    try:
        resp = requests.post(url, json=payload, headers=headers, timeout=timeout)
        resp.raise_for_status()
        routes = resp.json().get("routes", [])
        if not routes:
            return None
        dist_km = routes[0].get("distanceMeters", 0) / 1000.0
        dur_sec = int(float(routes[0].get("duration", "0s").rstrip("s")))
        return round(dist_km, 2), dur_sec
    except Exception as exc:
        print(f"[Google Maps API Warning] {origin} -> {destination}: {exc}")
        return None

def sync_graph_with_google_maps(api_key: Optional[str] = None) -> None:
    if not (api_key or os.getenv("GOOGLE_MAPS_API_KEY")):
        print("[Info] GOOGLE_MAPS_API_KEY not set; retaining baseline exercise distances.")
        return

    for u, neighbors in highway_graph.items():
        for v in neighbors:
            result = fetch_google_maps_traffic_distance_km(u, v, api_key=api_key)
            if result:
                dist_km, dur_sec = result
                update_edge_distance(u, v, dist_km, source=f"Google Maps TRAFFIC_AWARE (~{dur_sec // 60}m)")

KENHA_NOTICES_URL = "https://kenha.co.ke/public-other-notices/"
CAPITAL_FM_SEARCH_URL = "https://capitalfm.africa/?s=kenha+closure+diversion+traffic+disruption"

def _extract_mentioned_towns(text: str) -> List[str]:
    found: List[str] = []
    lower_text = text.lower()
    for town in CORRIDOR_TOWNS:
        if re.search(r"\b" + re.escape(town.lower()) + r"\b", lower_text):
            norm = normalize_town_name(town)
            if norm not in found:
                found.append(norm)
    return found

def _classify_disruption_status(text: str) -> Optional[str]:
    t = text.lower()
    if any(k in t for k in ["closure", "closed", "diversion", "diverted", "blocked", "landslide", "washout"]):
        return "Blocked"
    if any(k in t for k in ["disruption", "traffic", "advisory", "maintenance", "construction", "caution", "slow"]):
        return "Caution"
    return None

def scrape_kenha_notices(timeout: int = 12) -> List[Dict[str, object]]:
    findings: List[Dict[str, object]] = []
    try:
        resp = requests.get(KENHA_NOTICES_URL, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for el in soup.find_all(["tr", "article", "li", "h2", "h3", "h4"]):
            text = " ".join(el.get_text(" ", strip=True).split())
            if 12 <= len(text) <= 500:
                status = _classify_disruption_status(text)
                towns = _extract_mentioned_towns(text)
                if status and towns:
                    findings.append({
                        "source": "KeNHA Public Notices",
                        "status": status,
                        "towns": towns,
                        "headline": text[:220],
                    })
    except Exception as exc:
        print(f"[KeNHA Scraper Notice] Could not fetch live page ({exc}).")
    return findings

def scrape_capital_fm_disruptions(timeout: int = 12) -> List[Dict[str, object]]:
    findings: List[Dict[str, object]] = []
    try:
        resp = requests.get(CAPITAL_FM_SEARCH_URL, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for article in soup.find_all("article"):
            text = " ".join(article.get_text(" ", strip=True).split())
            if 15 <= len(text) <= 500:
                status = _classify_disruption_status(text)
                towns = _extract_mentioned_towns(text)
                if status and towns:
                    findings.append({
                        "source": "Capital FM Search",
                        "status": status,
                        "towns": towns,
                        "headline": text[:220],
                    })
    except Exception as exc:
        print(f"[Capital FM Scraper Notice] Could not fetch live page ({exc}).")
    return findings

def ingest_scraped_advisories_into_kb() -> None:
    notices = scrape_kenha_notices() + scrape_capital_fm_disruptions()
    if not notices:
        print("[KB Ingestion] No matching corridor disruption advisories parsed from live pages.")
        return

    for item in notices:
        mentioned = set(item["towns"])  # type: ignore[arg-type]
        for u, neighbors in highway_graph.items():
            for v in neighbors:
                if u in mentioned and v in mentioned:
                    penalty = 15.0 if item["status"] == "Caution" else 0.0
                    add_road_fact(
                        src=u,
                        dst=v,
                        status=str(item["status"]),
                        reason=str(item["headline"]),
                        penalty_km=penalty,
                        source=str(item["source"]),
                    )

# Example programmatic fact addition
add_road_fact("Nakuru", "Kericho", "Caution", "Escarpment / highland mist & heavy trailer load", 0.0)
show_kb()

Added fact: Road(Nakuru -> Kericho) = Caution (Escarpment / highland mist & heavy trailer load) [Manual/KB]

Current Trailer Routing Knowledge Base
Total Origin Towns: 28 | Corridor Authority: Kenya National Highways Authority (KeNHA)
Active Road Condition Facts:
  Nakuru -> Kericho : Caution (+0.0 km) | Escarpment / highland mist & heavy trailer load
Domain Rules:
  Blocked  -> Exclude segment from candidate search paths; reroute via next lowest-cost path.
  Caution  -> Keep segment in search but flag extra trailer braking/stability caution.
  Default  -> Traverse segment using shortest cumulative distance (Uniform Cost Search).
--------------------------------------------


In [4]:
# Steps 6 & 7: Encode Specific Problem Instance & Pose Queries for Answers
def ucs_with_kb(
    graph: Dict[str, Dict[str, float]],
    conditions: Dict[Tuple[str, str], Dict[str, object]],
    start: str = "Nairobi",
    goal: str = "Kisumu",
):
    start = normalize_town_name(start)
    goal = normalize_town_name(goal)
    frontier = [(0.0, start, [start], [])]
    best_cost: Dict[str, float] = {start: 0.0}

    while frontier:
        cost, node, path, advisories = heapq.heappop(frontier)

        if node == goal:
            return {
                "reachable": True,
                "path": path,
                "total_cost_km": round(cost, 2),
                "advisories": advisories,
            }

        for neighbor, base_dist in graph.get(node, {}).items():
            cond = conditions.get((node, neighbor))
            if cond and cond["status"] == "Blocked":
                continue

            step_penalty = float(cond["penalty_km"]) if (cond and cond["status"] == "Caution") else 0.0
            new_cost = cost + base_dist + step_penalty
            new_advisories = list(advisories)
            if cond and cond["status"] == "Caution":
                new_advisories.append(f"{node}->{neighbor}: {cond['reason']}")

            if neighbor not in best_cost or new_cost < best_cost[neighbor]:
                best_cost[neighbor] = new_cost
                heapq.heappush(frontier, (new_cost, neighbor, path + [neighbor], new_advisories))

    return {"reachable": False, "path": [], "total_cost_km": float("inf"), "advisories": []}

def query_agent(start: str = "Nairobi", goal: str = "Kisumu"):
    result = ucs_with_kb(highway_graph, road_conditions, start, goal)
    print(f"\n=== Query: Optimal Trailer Route ({start} -> {goal}) ===")
    if not result["reachable"]:
        print("Agent Decision: No valid route found under current KB constraints.")
        return result
    print("Optimal Path:", " -> ".join(result["path"]))
    print(f"Total Path Cost: {result['total_cost_km']} km")
    print("Route Advisories:", "; ".join(result["advisories"]) if result["advisories"] else "None")
    return result

#execution
query_agent("Nairobi", "Kisumu")


=== Query: Optimal Trailer Route (Nairobi -> Kisumu) ===
Optimal Path: Nairobi -> Ruaka -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Kericho -> Ahero -> Kisumu
Total Path Cost: 357.0 km
Route Advisories: Nakuru->Kericho: Escarpment / highland mist & heavy trailer load


{'reachable': True,
 'path': ['Nairobi',
  'Ruaka',
  'Limuru',
  'Naivasha',
  'Gilgil',
  'Nakuru',
  'Kericho',
  'Ahero',
  'Kisumu'],
 'total_cost_km': 357.0,
 'advisories': ['Nakuru->Kericho: Escarpment / highland mist & heavy trailer load']}

In [5]:
#Step 8:Querying the Knowledge Base
#Pose multiple queries to test sub-routes and alternative origins
query_agent("Nakuru", "Kisumu")
query_agent("Nairobi", "Eldoret")
query_agent("Maai-Mahiu", "Kisumu")


=== Query: Optimal Trailer Route (Nakuru -> Kisumu) ===
Optimal Path: Nakuru -> Kericho -> Ahero -> Kisumu
Total Path Cost: 191.4 km
Route Advisories: Nakuru->Kericho: Escarpment / highland mist & heavy trailer load

=== Query: Optimal Trailer Route (Nairobi -> Eldoret) ===
Optimal Path: Nairobi -> Ruaka -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Timboroa -> Eldoret
Total Path Cost: 321.8 km
Route Advisories: None

=== Query: Optimal Trailer Route (Maai-Mahiu -> Kisumu) ===
Optimal Path: Maai-Mahiu -> Narok -> Bomet -> Kisii -> Awach -> Ahero -> Kisumu
Total Path Cost: 367.5 km
Route Advisories: None


{'reachable': True,
 'path': ['Maai-Mahiu', 'Narok', 'Bomet', 'Kisii', 'Awach', 'Ahero', 'Kisumu'],
 'total_cost_km': 367.5,
 'advisories': []}

In [6]:
#Step 9: Debugging the Knowledge Base
# Simulate a road closure on a key segment of the optimal route and observe KB adaptation
print("\nDebugging Scenario 1: Blocking 'Kericho -> Ahero' ")
add_road_fact("Kericho", "Ahero", "Blocked", "Landslide / road maintenance closure")
query_agent("Nairobi", "Kisumu")

# Restore segment and test unknown/unconnected town query
print("\nDebugging Scenario 2: Clearing blockage & testing unreachable node ")
road_conditions.pop(("Kericho", "Ahero"), None)
query_agent("Kisumu", "Nairobi")  # Directed graph check (edges are unidirectional)


Debugging Scenario 1: Blocking 'Kericho -> Ahero' 
Added fact: Road(Kericho -> Ahero) = Blocked (Landslide / road maintenance closure) [Manual/KB]

=== Query: Optimal Trailer Route (Nairobi -> Kisumu) ===
Optimal Path: Nairobi -> Maai-Mahiu -> Narok -> Bomet -> Kisii -> Awach -> Ahero -> Kisumu
Total Path Cost: 421.3 km
Route Advisories: None

Debugging Scenario 2: Clearing blockage & testing unreachable node 

=== Query: Optimal Trailer Route (Kisumu -> Nairobi) ===
Agent Decision: No valid route found under current KB constraints.


{'reachable': False, 'path': [], 'total_cost_km': inf, 'advisories': []}

**Step 10: Final Reflection (Challenges & Recommended Improvements)**

*Directed vs. Undirected Graph Assumption*: The baseline graph remains strictly directed toward Kisumu, and integrating Google Maps or scrapers does not create reverse edges; bidirectional routing or backward rerouting still requires explicit reverse edges or reverse queries.

*Static Distance vs. Trailer Dynamics*: While the agent now supports traffic-aware routing updates via the Google Maps Routes API and advisory penalties, it still lacks trailer-specific factors like payload weight, escarpment gradients, axle-load limits, and travel-time weighting in UCS.

*Consistency & Conflict Resolution*: Because road_conditions currently overwrites entries on update and maps scraped town mentions onto graph edges, adding timestamps, source confidence/precedence rules, and stale-fact cleanup is a top priority.